# Legal-BERT Multi-Label Misconduct Classification

Fine-tunes `nlpaueb/legal-bert-base-uncased` for multi-label classification of
health practitioner misconduct types.

**Setup:** Upload `data/annotations/splits/` (train.jsonl, val.jsonl, test.jsonl) to
your Google Drive under `health_misconduct_nlp/splits/`.

**Categories (8):** CLIN, PRESC, BOUND, IMPR, FRAUD, COMM, PROF, N/A

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
import json
import os
import re
from pathlib import Path

import numpy as np
import torch
from google.colab import drive
from sklearn.metrics import classification_report, f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# Mount Google Drive
drive.mount('/content/drive')

# Paths — adjust if your Drive folder structure differs
DRIVE_DIR = Path('/content/drive/MyDrive/health_misconduct_nlp')
SPLITS_DIR = DRIVE_DIR / 'splits'
MODEL_DIR = DRIVE_DIR / 'models' / 'legal_bert_misconduct'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
MODEL_NAME = 'nlpaueb/legal-bert-base-uncased'
MAX_LENGTH = 512
BATCH_SIZE = 8
NUM_EPOCHS = 15
LEARNING_RATE = 5e-5  # increased from 2e-5 for small dataset
THRESHOLD = 0.3       # lowered from 0.5 — small dataset makes sigmoid outputs conservative
SEED = 42

MISCONDUCT_L1_CODES = ['CLIN', 'PRESC', 'BOUND', 'IMPR', 'FRAUD', 'COMM', 'PROF', 'N/A']
NUM_LABELS = len(MISCONDUCT_L1_CODES)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Load data
def load_split(name):
    records = []
    with open(SPLITS_DIR / f'{name}.jsonl') as f:
        for line in f:
            records.append(json.loads(line))
    return records

train_data = load_split('train')
val_data = load_split('val')
test_data = load_split('test')

# Preprocess: use the LAST ~3000 chars (findings/orders section) instead of
# the first 512 tokens. Legal decisions state misconduct types and outcomes
# near the end; the beginning is procedural preamble.
ORDERS_RE = re.compile(
    r'(?:^|\n)\s*(?:ORDERS?|THE\s+ORDERS?|DETERMINATION|DECISION|FINDINGS?)\s*\n',
    re.IGNORECASE | re.MULTILINE,
)

def extract_key_sections(text, max_chars=3000):
    """Extract findings/orders sections, falling back to last N chars."""
    matches = list(ORDERS_RE.finditer(text))
    if matches:
        # Take from the first findings/orders heading onward
        section_text = text[matches[0].start():]
        if len(section_text) > max_chars:
            section_text = section_text[:max_chars]
        return section_text
    # Fallback: last max_chars characters
    return text[-max_chars:]

for record in train_data + val_data + test_data:
    record['full_text'] = extract_key_sections(record['full_text'])

print(f'Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}')
print(f'Sample text length after preprocessing: {len(train_data[0]["full_text"])} chars')

In [ ]:
# Binarise labels
def binarise(labels):
    if labels is None:
        return [0.0] * NUM_LABELS
    if isinstance(labels, str):
        labels = json.loads(labels.replace("'", '"'))
    vec = [0.0] * NUM_LABELS
    for label in labels:
        if label in MISCONDUCT_L1_CODES:
            vec[MISCONDUCT_L1_CODES.index(label)] = 1.0
    return vec


class MisconductDataset(Dataset):
    def __init__(self, records, tokenizer, max_length=MAX_LENGTH):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        encoding = self.tokenizer(
            record['full_text'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        labels = binarise(record['misconduct_L1'])
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(labels, dtype=torch.float),
        }

In [ ]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type='multi_label_classification',
)

# Create datasets
train_dataset = MisconductDataset(train_data, tokenizer)
val_dataset = MisconductDataset(val_data, tokenizer)
test_dataset = MisconductDataset(test_data, tokenizer)

print(f'Model loaded: {MODEL_NAME}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Custom metrics for Trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = (torch.sigmoid(torch.tensor(logits)) > THRESHOLD).int().numpy()
    labels = labels.astype(int)

    macro_f1 = f1_score(labels, predictions, average='macro', zero_division=0)
    micro_f1 = f1_score(labels, predictions, average='micro', zero_division=0)

    per_label_f1 = f1_score(labels, predictions, average=None, zero_division=0)
    metrics = {
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
    }
    for i, code in enumerate(MISCONDUCT_L1_CODES):
        metrics[f'f1_{code}'] = per_label_f1[i]

    return metrics

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=10,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    report_to='none',
    save_total_limit=3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

# Train
print('Starting training...')
trainer.train()

In [ ]:
# Evaluate on test set
print('\n=== Test Set Evaluation ===')
test_results = trainer.predict(test_dataset)

logits = test_results.predictions
labels = test_results.label_ids.astype(int)
predictions = (torch.sigmoid(torch.tensor(logits)) > THRESHOLD).int().numpy()

print('\nPer-label classification report:')
print(classification_report(
    labels, predictions,
    target_names=MISCONDUCT_L1_CODES,
    zero_division=0,
))

macro_f1 = f1_score(labels, predictions, average='macro', zero_division=0)
micro_f1 = f1_score(labels, predictions, average='micro', zero_division=0)
print(f'Macro F1: {macro_f1:.4f}')
print(f'Micro F1: {micro_f1:.4f}')

In [ ]:
# Save the best model
final_model_dir = MODEL_DIR / 'final'
trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))
print(f'Model saved to {final_model_dir}')

In [ ]:
# Predict on full corpus (if uploaded)
CORPUS_PATH = DRIVE_DIR / 'all_decisions.jsonl'

if CORPUS_PATH.exists():
    print('Loading full corpus...')
    corpus_records = []
    with open(CORPUS_PATH) as f:
        for line in f:
            corpus_records.append(json.loads(line))
    print(f'  {len(corpus_records)} decisions loaded')

    # Preprocess texts the same way as training
    for record in corpus_records:
        record['_input_text'] = extract_key_sections(record['full_text'])

    # Predict in batches
    all_predictions = []
    model.eval()
    model.to(device)

    for i in range(0, len(corpus_records), BATCH_SIZE):
        batch = corpus_records[i:i + BATCH_SIZE]
        texts = [r['_input_text'] for r in batch]
        encoding = tokenizer(
            texts,
            max_length=MAX_LENGTH,
            padding=True,
            truncation=True,
            return_tensors='pt',
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoding)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()

        for j, record in enumerate(batch):
            pred_labels = [
                MISCONDUCT_L1_CODES[k]
                for k in range(NUM_LABELS)
                if probs[j, k] > THRESHOLD
            ]
            all_predictions.append({
                'id': record['id'],
                'predicted_misconduct_L1_bert': pred_labels,
                'misconduct_probs': {MISCONDUCT_L1_CODES[k]: float(probs[j, k]) for k in range(NUM_LABELS)},
            })

        if (i // BATCH_SIZE) % 50 == 0:
            print(f'  Processed {i + len(batch)}/{len(corpus_records)}')

    # Save predictions
    output_path = DRIVE_DIR / 'misconduct_predictions_bert.jsonl'
    with open(output_path, 'w') as f:
        for pred in all_predictions:
            f.write(json.dumps(pred) + '\n')
    print(f'\nPredictions saved to {output_path}')

    # Summary
    from collections import Counter
    label_counts = Counter()
    for p in all_predictions:
        label_counts.update(p['predicted_misconduct_L1_bert'])
    print('\nBERT prediction distribution:')
    for code in MISCONDUCT_L1_CODES:
        count = label_counts.get(code, 0)
        print(f'  {code}: {count} ({count/len(all_predictions)*100:.1f}%)')
else:
    print(f'Full corpus not found at {CORPUS_PATH}')
    print('Upload all_decisions.jsonl to your Drive to predict on full corpus.')

In [ ]:
# Compare sklearn vs BERT (if sklearn predictions available)
SKLEARN_PATH = DRIVE_DIR / 'misconduct_predictions_sklearn.jsonl'

if SKLEARN_PATH.exists() and CORPUS_PATH.exists():
    sklearn_preds = {}
    with open(SKLEARN_PATH) as f:
        for line in f:
            r = json.loads(line)
            sklearn_preds[r['id']] = set(r['predicted_misconduct_L1'])

    bert_preds = {}
    for p in all_predictions:
        bert_preds[p['id']] = set(p['predicted_misconduct_L1_bert'])

    # Agreement
    common_ids = set(sklearn_preds.keys()) & set(bert_preds.keys())
    agree = sum(1 for id in common_ids if sklearn_preds[id] == bert_preds[id])
    print(f'sklearn vs BERT agreement: {agree}/{len(common_ids)} ({agree/len(common_ids)*100:.1f}%)')
else:
    print('sklearn predictions not found. Upload misconduct_predictions_sklearn.jsonl to compare.')